In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('final_plots/comet_65k_by_layers.csv')

In [ ]:
cols_to_plot = [
    'sweep_pair.num_heads',
    'sweep_pair.num_mkeys',
    'sweep_pair.num_nkeys',
    'sweep_pair.dict_size',
    'explained_variance',
    'parameters_count',
    'accum_num_flops',
    'n_tokens'
]


In [ ]:
mapping = {
    'sweep id': ("1.5B", 1000 * 1e6),
    'sweep id': ("1.5B", 500 * 1e6),
    'sweep id': ("1.5B", 100 * 1e6),
    'sweep id': ("3B", 1000 * 1e6),
    'sweep id': ("3B", 500 * 1e6),
    'sweep id': ("3B", 100 * 1e6),
    'sweep id': ("7B", 500 * 1e6),
    'sweep id': ("7B", 100 * 1e6)
}

In [ ]:
qwen_data = df[df['model_name'] == 'Qwen/Qwen2.5-1.5B']
gemma_data = df[df['model_name'] == 'google/gemma-2-2b']

In [ ]:
data = []
for i, row in df.iterrows():
    entry = {
        'Name': row['Name'],
        'H': row['sweep_pair.num_heads'],
        'M': row['sweep_pair.num_mkeys'],
        'N': row['sweep_pair.num_nkeys'],
        'F': row['sweep_pair.dict_size'],
        'Explained Variance': row['explained_variance'],
        'Parameters': row['parameters_count'],
        'FLOPS': row['accum_num_flops'],
        'Tokens': row['n_tokens'],
        'Layer': row['layer']
    }

    if 'kronsae' in row['Name']:
        entry['SAE'] = 'KronSAE'
    else:
        entry['SAE'] = 'TopK'
    
    data.append(entry)

data = pd.DataFrame(data)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4), dpi=250)
ax.grid(alpha=0.35)
sns.lineplot(data, x='Layer', y='Explained Variance', hue='SAE', ax=ax,palette='tab10', marker='o',)

ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Explained Variance', fontsize=12)
plt.legend()
ax.set_xticks(data['Layer'].unique())
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
plt.tight_layout()
plt.savefig('layer vs explained variance.pdf')
plt.show()

In [ ]:
df = pd.read_csv('final_plots/ev_l0.csv')

In [ ]:
kronsae = []
topk = []
for i, row in df.iterrows():
    entry = {
        'Name': row['Name'],
        'H': row['sweep_pair.num_heads'],
        'M': row['sweep_pair.num_mkeys'],
        'N': row['sweep_pair.num_nkeys'],
        'F': row['sweep_pair.dict_size'],
        'Explained Variance': row['explained_variance'],
        'Parameters': row['parameters_count'],
        'FLOPS': row['accum_num_flops'],
        'Tokens': row['n_tokens'],
        'Model': row['model_name'],
        'L0': row['topk']
    }
    
    if 'kronsae' in row['Name']:
        kronsae.append(entry)
    else:
        topk.append(entry)

kronsae = pd.DataFrame(kronsae)
topk = pd.DataFrame(topk)

In [ ]:
qwen_kronsae_selected = kronsae[(kronsae["Model"] == "Qwen/Qwen2.5-1.5B") & (kronsae["M"] == 2) & (kronsae["N"] == 8)].sort_values("L0")

In [ ]:
google_kronsae_selected = kronsae[(kronsae["Model"] == "google/gemma-2-2b") & (kronsae["M"] == 2) & (kronsae["N"] == 8)].sort_values("Explained Variance")

In [ ]:
merged = pd.concat([qwen_kronsae_selected, google_kronsae_selected])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.lines import Line2D 

fig, ax = plt.subplots(figsize=(5, 4), dpi=250)
cplt = sns.color_palette('tab10')

sns.lineplot(
    data=topk[(topk["Model"] == "google/gemma-2-2b")],
    x='L0',
    y='Explained Variance',
    marker='o',
    lw=1.2,
    color=cplt[0],
    ls='--',
    markersize=10,
)

sns.lineplot(
    data=merged[(merged["Model"] == "google/gemma-2-2b")],
    x='L0',
    y='Explained Variance',
    markers=True,
    marker='^',
    lw=1.2,
    color=cplt[1],
    ls='--',
    markersize=10,
)

sns.lineplot(
    data=topk[(topk["Model"] == "Qwen/Qwen2.5-1.5B")],
    x='L0', 
    y='Explained Variance',
    marker='o',
    lw=1.2,
    color=cplt[0],
    ls='-',
    markersize=10,
)



sns.lineplot(
    data=merged[(merged["Model"] == "Qwen/Qwen2.5-1.5B")],
    x='L0',
    y='Explained Variance',
    markers=True,
    marker='^',
    lw=1.2,
    color=cplt[1],
    ls='-',
    markersize=10,
)

ax.set_xscale('log', base=2)
ax.set_xticks(topk['L0'].unique())
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())

ax.set_xlabel(r'$\ell_0$ Sparsity', fontsize=12)
ax.set_ylabel('Explained Variance', fontsize=12)
ax.grid(True, alpha=0.35, which='both')

handles, labels = ax.get_legend_handles_labels()
star_marker = Line2D(
    [0], [0], color= cplt[0], marker='o', linestyle="", markersize=4, label='TopK'
)
kronsae_marker = Line2D(
    [0], [0], color= cplt[1], marker='^', linestyle="", markersize=4, label='TopK'
)
qwen_marker = Line2D(
    [0], [0], color='gray', marker='', linestyle="-", markersize=4, label='TopK'
)

gemma_marker = Line2D(
    [0], [0], color='gray', marker='', linestyle="--", markersize=4, label='TopK'
)

custom_handles =  [star_marker, kronsae_marker]  + [qwen_marker, gemma_marker]
custom_labels = ['TopK', 'KronSAE', 'Qwen2.5 1.5B', 'Gemma 2 2B', ]

ax.legend(
    handles=custom_handles,
    labels=custom_labels,
    loc='upper left',
)

plt.tight_layout()
plt.savefig(f'l0 vs explained variance.pdf')
plt.show()